# Notebook 05 — Evaluation

**Goal:** Measure the trained model's accuracy on the held-out test set and
produce an error analysis focused on Qaloon-specific phonological patterns.

**Why this notebook exists:**
Aggregate WER/CER numbers tell us whether training succeeded, but they don't
tell us *where* the model fails.  For a Qaloon recitation checker we care most
about diacritical errors — a fatha/kasra confusion matters more than a word
boundary error.  This notebook computes both aggregate metrics and a breakdown
by error category.

**Output:**
- Printed WER / CER
- `Documentation/evaluation_report.json`

## Step 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/iqraa-ai')
!pip install -q -r /content/drive/MyDrive/iqraa-ai/requirements.txt

## Step 1 — Configure paths

In [ ]:
from pathlib import Path

REPO_ROOT   = Path('/content/drive/MyDrive/iqraa-ai')
MODEL_DIR   = REPO_ROOT / 'models' / 'qaloon_wav2vec2'
DATASET_DIR = REPO_ROOT / 'data' / 'processed' / 'hf_dataset'
REPORT_OUT  = REPO_ROOT / 'Documentation' / 'evaluation_report.json'

## Step 2 — Run evaluation

We call `src.evaluate.evaluate_dataset()` which iterates over the test split,
transcribes each segment, and computes WER and CER with `jiwer`.

In [ ]:
from src.evaluate import evaluate_dataset

metrics = evaluate_dataset(
    model_dir=str(MODEL_DIR),
    dataset_dir=str(DATASET_DIR),
    output_path=str(REPORT_OUT),
)

print(f"WER : {metrics['wer']:.2%}")
print(f"CER : {metrics['cer']:.2%}")
print(f"N   : {metrics['num_samples']} test samples")

## Step 3 — Manual error inspection

Print side-by-side comparisons of reference vs. hypothesis for the 20 worst
samples (by CER).  This reveals systematic error patterns — e.g., if the model
consistently drops shadda or confuses fatha/damma, those rules need more data
or a targeted augmentation pass.

In [ ]:
import jiwer
import torch
from src.dataset import load_dataset_from_disk
from src.evaluate import load_model_and_processor, transcribe_batch

model, processor = load_model_and_processor(str(MODEL_DIR))
ds = load_dataset_from_disk(str(DATASET_DIR))
test = ds['test']

errors = []
for sample in test:
    audio = sample['audio']
    hyp   = transcribe_batch([audio['array']], [audio['sampling_rate']],
                             model, processor)[0]
    cer   = jiwer.cer(sample['text'], hyp)
    errors.append({'id': sample['id'], 'ref': sample['text'],
                   'hyp': hyp, 'cer': cer})

errors.sort(key=lambda x: x['cer'], reverse=True)

print('Top 20 worst segments:')
for e in errors[:20]:
    print(f"\n[{e['id']}] CER={e['cer']:.2%}")
    print(f"  REF: {e['ref']}")
    print(f"  HYP: {e['hyp']}")